# Visualize Correlations with HXK1 Protein Abundance: REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import chain, combinations, product
import re
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    PROTEIN_ID,
    RAW_DATA_DIR,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import identify_independent_snps
from hk1_r12ter_analysis.util import ceil_to_decimal
from hk1_r12ter_analysis.visualize import (
    plot_correlations_vs_pvalues,
    plot_hemolysis_pvalues_for_snp,
)

## REDS Recall

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
main_data_type = f"Proteomics-{PROTEIN_ID}"
norm_str = "median-none-auto"
correlation_statistic = "Spearman"

filetype = "pdf"

variable_fancy_replacements = {
    "Index.Adjusted.Osmotic.Hemolysis": "Osmotic Hemolysis",
    "Index.Adjusted.Storage.Hemolysis": "Storage Hemolysis",
    "Index.Adjusted.Oxidative.Hemolysis": "Oxidative Hemolysis",
    source_type: source_type.replace("-", " "),
}

### Load correlation and statistics

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
input_filename = make_filename(source_type, f"{main_data_type}_All_Data", correlation_statistic)
df_correlations_tall = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=None, low_memory=False
)

group_value = "ALL"
correlations_options = {
    "Group": group_value,
    "DataType1": "Proteomics",
}
for key, value in correlations_options.items():
    df_correlations_tall = (
        df_correlations_tall[df_correlations_tall[key] == value]
        .drop_duplicates()
        .drop(key, axis=1)
    )
df_correlations_tall = df_correlations_tall.set_index(["DataType2", "Variable2"]).drop(
    "Variable1", axis=1
)
df_correlations_tall.index.names = ["DataType", "Variable"]
df_correlations_tall

### Visualize correlations
#### Genomics

In [ ]:
other_data_type = f"Genomics-{GENE_ID}"
data = df_correlations_tall.copy()
data = df_correlations_tall.loc[
    pd.IndexSlice[other_data_type, :],
    [f"{correlation_statistic} statistic", f"{correlation_statistic} -log10(p-value)"],
]
data = data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)

group_options = {
    "ALL": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 5,
        "ydecimals": 2,
        "ymajor_tick_multiple": 1,
    },
    "10": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 0.5,
    },
    "23": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 0.5,
    },
    "42": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 0.5,
    },
}

group_value_kwargs = group_options[group_value]
xlim_pads = (0.0, 0.125)
ylim_pads = (0.02, 0.02)

fig, ax = plt.subplots(figsize=(4, 4), nrows=1, ncols=1)
cax, ax = (None, ax) if isinstance(ax, mpl.axes.Axes) else ax
title = "\n".join(
    [
        f"{PROTEIN_ID} Protein Abundance",
        f"Correlates with {GENE_ID} SNPs in ",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)
xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)

xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
xlim = (-xabsmax * (1 + xlim_pads[0]), xabsmax * (1 + xlim_pads[1]))
print(f"x-limits: {xlim[0]:.4f}, {xlim[1]:.4f}")
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")

ax, cax = plot_correlations_vs_pvalues(
    data=data,
    correlation_statistic=data.columns[0],
    neg_log10_pvalue=data.columns[1],
    ax=ax,
    s=40,
    fontdict=dict(size="large"),
    title=title,
    xaxis_kwargs=dict(
        lim=xlim,
        label=f"Spearman correlation ($\\rho$)",
        major_tick_multiple=group_value_kwargs["xmajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    yaxis_kwargs=dict(
        lim=ylim,
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
        which="both",
    ),
    cax=cax,
)
sns.despine(ax=ax)

output_dirpath = REPORTS_DIR / "REDS" / "Correlations" / main_data_type
# output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [source_type, f"{correlation_statistic}Correlations", other_data_type]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
output_dirpath.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dirpath / output_filename)
print(output_dirpath / output_filename)
plt.show()
plt.close()
data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False).head(10)

#### Proteomics

In [ ]:
other_data_type = "Proteomics"
data = df_correlations_tall.copy()
data = df_correlations_tall.loc[
    pd.IndexSlice[other_data_type, :],
    [f"{correlation_statistic} statistic", f"{correlation_statistic} -log10(p-value)"],
]
data = data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)

group_options = {
    "ALL": {
        "xscalar": 10,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.2,
        "yscalar": 10,
        "ydecimals": 0,
        "ymajor_tick_multiple": 10,
    },
    "10": {
        "xscalar": 10,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.2,
        "yscalar": 5,
        "ydecimals": 0,
        "ymajor_tick_multiple": 10,
    },
    "23": {
        "xscalar": 10,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.2,
        "yscalar": 5,
        "ydecimals": 0,
        "ymajor_tick_multiple": 10,
    },
    "42": {
        "xscalar": 10,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.2,
        "yscalar": 5,
        "ydecimals": 0,
        "ymajor_tick_multiple": 10,
    },
}
group_value_kwargs = group_options[group_value]
xlim_pads = (0.0, 0.125)
ylim_pads = (0.02, 0.0)

fig, ax = plt.subplots(figsize=(4, 4), nrows=1, ncols=1)
cax, ax = (None, ax) if isinstance(ax, mpl.axes.Axes) else ax
title = "\n".join(
    [
        f"{PROTEIN_ID} Protein Abundance",
        f"Correlates with {other_data_type} in ",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)

xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
xlim = (-xabsmax * (1 + xlim_pads[0]), xabsmax * (1 + xlim_pads[1]))
print(f"x-limits: {xlim[0]:.4f}, {xlim[1]:.4f}")
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")

ax, cax = plot_correlations_vs_pvalues(
    data=data,
    correlation_statistic=data.columns[0],
    neg_log10_pvalue=data.columns[1],
    ax=ax,
    s=40,
    fontdict=dict(size="large"),
    title=title,
    xaxis_kwargs=dict(
        lim=xlim,
        label=f"Spearman correlation ($\\rho$)",
        major_tick_multiple=group_value_kwargs["xmajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    yaxis_kwargs=dict(
        lim=ylim,
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
        which="both",
    ),
    cax=cax,
)
sns.despine(ax=ax)

output_dirpath = REPORTS_DIR / "REDS" / "Correlations" / main_data_type
# output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [source_type, f"{correlation_statistic}Correlations", other_data_type]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
output_dirpath.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dirpath / output_filename)
print(output_dirpath / output_filename)
plt.show()
plt.close()
data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False).head(10)

#### Metabolomics

In [ ]:
other_data_type = "Metabolomics"
data = df_correlations_tall.copy()
data = df_correlations_tall.loc[
    pd.IndexSlice[other_data_type, :],
    [f"{correlation_statistic} statistic", f"{correlation_statistic} -log10(p-value)"],
]
data = data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)

group_options = {
    "ALL": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 1,
        "ydecimals": 0,
        "ymajor_tick_multiple": 1,
    },
    "10": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
    "23": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
    "42": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
}
group_value_kwargs = group_options[group_value]
xlim_pads = (0.0, 0.125)
ylim_pads = (0.02, 0.0)

fig, ax = plt.subplots(figsize=(4, 4), nrows=1, ncols=1)
cax, ax = (None, ax) if isinstance(ax, mpl.axes.Axes) else ax
title = "\n".join(
    [
        f"{PROTEIN_ID} Protein Abundance",
        f"Correlates with {other_data_type} in ",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)

xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
xlim = (-xabsmax * (1 + xlim_pads[0]), xabsmax * (1 + xlim_pads[1]))
print(f"x-limits: {xlim[0]:.4f}, {xlim[1]:.4f}")
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")

ax, cax = plot_correlations_vs_pvalues(
    data=data,
    correlation_statistic=data.columns[0],
    neg_log10_pvalue=data.columns[1],
    ax=ax,
    s=40,
    fontdict=dict(size="large"),
    title=title,
    xaxis_kwargs=dict(
        lim=xlim,
        label=f"Spearman correlation ($\\rho$)",
        major_tick_multiple=group_value_kwargs["xmajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    yaxis_kwargs=dict(
        lim=ylim,
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
        which="both",
    ),
    cax=cax,
)
sns.despine(ax=ax)

output_dirpath = REPORTS_DIR / "REDS" / "Correlations" / main_data_type
# output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [source_type, f"{correlation_statistic}Correlations", other_data_type]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
output_dirpath.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dirpath / output_filename)
print(output_dirpath / output_filename)
plt.show()
plt.close()
data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False).head(10)

#### Lipidomics

In [ ]:
other_data_type = "Lipidomics"
data = df_correlations_tall.copy()
data = df_correlations_tall.loc[
    pd.IndexSlice[other_data_type, :],
    [f"{correlation_statistic} statistic", f"{correlation_statistic} -log10(p-value)"],
]
data = data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False)

group_options = {
    "ALL": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.04,
        "yscalar": 2,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
    "10": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 1,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
    "23": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 1,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
    "42": {
        "xscalar": 2,
        "xdecimals": 2,
        "xmajor_tick_multiple": 0.06,
        "yscalar": 1,
        "ydecimals": 1,
        "ymajor_tick_multiple": 1,
    },
}
group_value_kwargs = group_options[group_value]
xlim_pads = (0.0, 0.125)
ylim_pads = (0.02, 0.02)

fig, ax = plt.subplots(figsize=(4, 4), nrows=1, ncols=1)
cax, ax = (None, ax) if isinstance(ax, mpl.axes.Axes) else ax
title = "\n".join(
    [
        f"{PROTEIN_ID} Protein Abundance",
        f"Correlates with {other_data_type} in ",
        f"{variable_fancy_replacements.get(source_type)} Donors"
        + (f", {group_key} {group_value}" if group_value != "ALL" else ""),
    ]
)

xabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} statistic"].abs().max(),
    decimals=group_value_kwargs["xdecimals"],
    scalar=group_value_kwargs["xscalar"],
)
xlim = (-xabsmax * (1 + xlim_pads[0]), xabsmax * (1 + xlim_pads[1]))
print(f"x-limits: {xlim[0]:.4f}, {xlim[1]:.4f}")
yabsmax = ceil_to_decimal(
    data[f"{correlation_statistic} -log10(p-value)"].abs().max(),
    decimals=group_value_kwargs["ydecimals"],
    scalar=group_value_kwargs["yscalar"],
)
ylim = (-yabsmax * ylim_pads[0], yabsmax * (1 + ylim_pads[1]))
print(f"y-limits: {ylim[0]:.4f}, {ylim[1]:.4f}")

ax, cax = plot_correlations_vs_pvalues(
    data=data,
    correlation_statistic=data.columns[0],
    neg_log10_pvalue=data.columns[1],
    ax=ax,
    s=40,
    fontdict=dict(size="large"),
    title=title,
    xaxis_kwargs=dict(
        lim=xlim,
        label=f"Spearman correlation ($\\rho$)",
        major_tick_multiple=group_value_kwargs["xmajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    yaxis_kwargs=dict(
        lim=ylim,
        label="-log$_{10}$($p$-value)",
        major_tick_multiple=group_value_kwargs["ymajor_tick_multiple"],
        minor_ticks=True,
        tick_labelsize="medium",
    ),
    grid_kwargs=dict(
        linestyle="-",
        color="xkcd:light grey",
        which="both",
    ),
    cax=cax,
)
sns.despine(ax=ax)

output_dirpath = REPORTS_DIR / "REDS" / "Correlations" / main_data_type
# output_dirpath.mkdir(parents=True, exist_ok=True)
filename_args = [source_type, f"{correlation_statistic}Correlations", other_data_type]
if group_value != "ALL":
    filename_args += [f"{group_key}{group_value}"]
output_filename = make_filename(*filename_args, filetype=filetype)
output_dirpath.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dirpath / output_filename)
print(output_dirpath / output_filename)
plt.show()
plt.close()
data.sort_values(by=f"{correlation_statistic} -log10(p-value)", ascending=False).head(10)